## 1. Configuration et Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
from datetime import datetime

## 2. Création de la Session Spark avec Delta Lake

In [2]:
# Configuration des chemins
jar_path = os.path.abspath("jars/delta-spark_2.12-3.2.1.jar") + "," + os.path.abspath("jars/delta-storage-3.2.1.jar")

spark = SparkSession.builder \
    .appName("Bronze_Pipeline_JSON_to_Delta") \
    .config("spark.jars", jar_path) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.streaming.schemaInference", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✓ Spark Session créée (version {spark.version})")

your 131072x1 screen size is bogus. expect trouble
25/12/16 12:13:22 WARN Utils: Your hostname, PCFlo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/16 12:13:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/12/16 12:13:22 WARN Utils: Your hostname, PCFlo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/16 12:13:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/12/16 12:13:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/16 12:13:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogL

✓ Spark Session créée (version 3.5.3)


## 3. Configuration des Chemins

In [3]:
# Chemins pour l'architecture Médaillon
INPUT_PATH = "data/sensor_data"
OUTPUT_PATH = "output/delta/bronze/sensor_data"
CHECKPOINT_PATH = "checkpoints/bronze"

print(f"📁 Input Path: {INPUT_PATH}")
print(f"📁 Output Path: {OUTPUT_PATH}")
print(f"📁 Checkpoint Path: {CHECKPOINT_PATH}")

📁 Input Path: data/sensor_data
📁 Output Path: output/delta/bronze/sensor_data
📁 Checkpoint Path: checkpoints/bronze


## 4. Schéma des Données Capteurs IoT

Structure des mesures de capteurs :
- `timestamp` : Horodatage ISO 8601
- `device_id` : Identifiant unique du capteur
- `building` : Bâtiment (A, B, C)
- `floor` : Étage
- `type` : Type de mesure (temperature, humidity, co2)
- `value` : Valeur mesurée
- `unit` : Unité (°C, %, ppm)

In [4]:
sensor_schema = StructType([
    StructField("timestamp", StringType(), False),
    StructField("device_id", StringType(), False),
    StructField("building", StringType(), False),
    StructField("floor", IntegerType(), False),
    StructField("type", StringType(), False),
    StructField("value", DoubleType(), False),
    StructField("unit", StringType(), False)
])

print("✓ Schéma défini")

✓ Schéma défini


## 5. Lecture du Flux JSON

Utilisation de `readStream` pour traiter les fichiers JSON de manière continue.

In [5]:
json_stream = spark.readStream \
    .schema(sensor_schema) \
    .option("maxFilesPerTrigger", 5) \
    .json(INPUT_PATH)

print("✓ Stream JSON configuré")
print(f"Schéma: {json_stream.schema}")

✓ Stream JSON configuré
Schéma: StructType([StructField('timestamp', StringType(), True), StructField('device_id', StringType(), True), StructField('building', StringType(), True), StructField('floor', IntegerType(), True), StructField('type', StringType(), True), StructField('value', DoubleType(), True), StructField('unit', StringType(), True)])


## 6. Transformations Bronze

Enrichissement des données brutes :
1. **Conversion timestamp** : String → Timestamp
2. **Horodatage ingestion** : Moment de traitement
3. **Source file** : Traçabilité du fichier source
4. **Détection anomalies** :
   - CO₂ > 1000 ppm
   - Température < 15°C ou > 30°C
   - Humidité < 20% ou > 80%
5. **Qualité des données** : Complétude des champs obligatoires

In [6]:
bronze_stream = json_stream \
    .withColumn("event_timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name()) \
    .withColumn(
        "anomaly_detected",
        when((col("type") == "co2") & (col("value") > 1000), True)
        .when((col("type") == "temperature") & ((col("value") < 15) | (col("value") > 30)), True)
        .when((col("type") == "humidity") & ((col("value") < 20) | (col("value") > 80)), True)
        .otherwise(False)
    ) \
    .withColumn(
        "data_quality",
        when(
            col("device_id").isNotNull() & 
            col("building").isNotNull() & 
            col("value").isNotNull(),
            "complete"
        ).otherwise("incomplete")
    ) \
    .select(
        col("device_id"),
        col("building"),
        col("floor"),
        col("type"),
        col("value"),
        col("unit"),
        col("event_timestamp"),
        col("ingestion_timestamp"),
        col("anomaly_detected"),
        col("data_quality"),
        col("source_file")
    )

print("✓ Transformations Bronze configurées")

✓ Transformations Bronze configurées


## 7. Écriture dans Delta Lake

Configuration du streaming vers Delta Lake :
- **Format** : Delta Lake (ACID + Versioning)
- **Mode** : Append (ajout continu)
- **Checkpoint** : Tolérance aux pannes
- **Trigger** : Traitement toutes les 10 secondes

In [ ]:
query = bronze_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime="10 seconds") \
    .start(OUTPUT_PATH)

print(f"✓ Streaming query démarré")
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")

✓ Streaming query démarré
Query ID: bdcbb461-9b56-43d3-9cd9-6f3ea6051073
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


25/12/16 12:13:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/12/16 12:13:59 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_99.json was not found. Was it deleted very recently?
25/12/16 12:13:59 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_76.json was not found. Was it deleted very recently?
25/12/16 12:13:59 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_60.json was not found. Was it deleted very recently?
25/12/16 12:13:59 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_37.json was not found. Was it deleted very recently?
25/12/16 12:13:59 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_102.json was not found. Was it deleted very recently?
25/12/16 12:13:59 ERROR MicroBatchExecution: Query [id = bdcbb461-9b56-43d3-9cd9-6f3ea6051073, runId = 9f6f73f4-0e58-45f5-bcc7-5cb6e50c4af6] terminated with error
java.lang.IllegalArgumentException: Wrong basePath data/sensor_data for the root path: file:/app/data/sensor_data/se

## 8. Surveillance du Streaming

Monitoring des métriques de traitement.

In [8]:
import time

# Attendre quelques secondes pour voir le traitement
time.sleep(20)

print("📊 Statut du streaming:")
print(f"Is Active: {query.isActive}")
print(f"Recent Progress: {len(query.recentProgress)} batches")

if query.recentProgress:
    latest = query.recentProgress[-1]
    print(f"\nDernier batch:")
    print(f"  - Batch ID: {latest.get('batchId', 'N/A')}")
    print(f"  - Input Rows: {latest.get('numInputRows', 0)}")
    print(f"  - Process Rate: {latest.get('processedRowsPerSecond', 0):.2f} rows/sec")

📊 Statut du streaming:
Is Active: False
Recent Progress: 0 batches


## 9. Vérification des Données Bronze

Lecture batch pour validation des données écrites.

In [9]:
# Lire les données Bronze
bronze_df = spark.read.format("delta").load(OUTPUT_PATH)

print(f"📊 Total enregistrements Bronze: {bronze_df.count()}")
print("\n📋 Aperçu des données:")
bronze_df.show(10, truncate=False)

25/12/16 12:14:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


📊 Total enregistrements Bronze: 500

📋 Aperçu des données:
+-----------------+--------+-----+------------------+-----+----+-------------------+-----------------------+----------------+------------+------------------------------------------------+
|device_id        |building|floor|type              |value|unit|event_timestamp    |ingestion_timestamp    |anomaly_detected|data_quality|source_file                                     |
+-----------------+--------+-----+------------------+-----+----+-------------------+-----------------------+----------------+------------+------------------------------------------------+
|sensor-co2-020   |A       |1    |co2               |901.0|ppm |2025-01-12 09:39:36|2025-12-16 11:38:50.012|false           |complete    |file:///app/data/sensor_data/sensor_data_91.json|
|sensor-temp-003  |B       |1    |temperature       |23.9 |°C  |2025-01-12 09:39:41|2025-12-16 11:38:50.012|false           |complete    |file:///app/data/sensor_data/sensor_data_91.json|
|

In [10]:
# Statistiques par bâtiment
print("📈 Statistiques par bâtiment:")
bronze_df.groupBy("building").agg(
    count("*").alias("total_records"),
    countDistinct("device_id").alias("unique_devices")
).show()

📈 Statistiques par bâtiment:
+--------+-------------+--------------+
|building|total_records|unique_devices|
+--------+-------------+--------------+
|       B|          256|             5|
|       A|          244|             5|
+--------+-------------+--------------+

+--------+-------------+--------------+
|building|total_records|unique_devices|
+--------+-------------+--------------+
|       B|          256|             5|
|       A|          244|             5|
+--------+-------------+--------------+



In [11]:
# Statistiques par type de capteur
print("📈 Statistiques par type de capteur:")
bronze_df.groupBy("type").agg(
    count("*").alias("total_records"),
    avg("value").alias("avg_value"),
    min("value").alias("min_value"),
    max("value").alias("max_value")
).show()

📈 Statistiques par type de capteur:
+------------------+-------------+------------------+---------+---------+
|              type|total_records|         avg_value|min_value|max_value|
+------------------+-------------+------------------+---------+---------+
|          humidity|           94| 44.78191489361702|     30.1|     59.6|
|       temperature|          215| 23.46511627906977|     18.1|     27.9|
|energy_consumption|           79|150.07848101265824|    102.4|    198.9|
|               co2|          112| 809.0267857142857|    400.0|   1200.0|
+------------------+-------------+------------------+---------+---------+

+------------------+-------------+------------------+---------+---------+
|              type|total_records|         avg_value|min_value|max_value|
+------------------+-------------+------------------+---------+---------+
|          humidity|           94| 44.78191489361702|     30.1|     59.6|
|       temperature|          215| 23.46511627906977|     18.1|     27.9|
|

In [12]:
# Anomalies détectées
print("⚠️  Anomalies détectées:")
anomalies = bronze_df.filter(col("anomaly_detected") == True)
print(f"Total anomalies: {anomalies.count()}")
anomalies.groupBy("type", "building").count().orderBy(desc("count")).show()

⚠️  Anomalies détectées:
Total anomalies: 33
Total anomalies: 33
+----+--------+-----+
|type|building|count|
+----+--------+-----+
| co2|       B|   19|
| co2|       A|   14|
+----+--------+-----+

+----+--------+-----+
|type|building|count|
+----+--------+-----+
| co2|       B|   19|
| co2|       A|   14|
+----+--------+-----+



## 10. Historique Delta Lake

Vérification du versioning et des transactions.

In [13]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, OUTPUT_PATH)
print("📜 Historique des transactions:")
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(10, truncate=False)

📜 Historique des transactions:
+-------+-----------------------+----------------+----------------------------------------------------------------------------------------+
|version|timestamp              |operation       |operationMetrics                                                                        |
+-------+-----------------------+----------------+----------------------------------------------------------------------------------------+
|19     |2025-12-16 11:40:20.724|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 18607, numAddedFiles -> 5}|
|18     |2025-12-16 11:40:10.7  |STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 18528, numAddedFiles -> 5}|
|17     |2025-12-16 11:40:00.782|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 18490, numAddedFiles -> 5}|
|16     |2025-12-16 11:39:50.82 |STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 18524, numAddedFi

## 11. Arrêt du Streaming

In [14]:
# Arrêter le streaming query
query.stop()
time.sleep(2)
print(f"✓ Streaming arrêté (Active: {query.isActive})")

✓ Streaming arrêté (Active: False)


## Concepts Clés Illustrés

### 1. Spark Structured Streaming
- Traitement de flux comme des tables infinies
- API unifiée pour batch et streaming

### 2. Checkpointing
- Sauvegarde de l'état du streaming
- Reprise exacte après échec
- Garantie de tolérance aux pannes

### 3. Mode Append
- Seules les nouvelles lignes sont écrites
- Idéal pour données immuables (Bronze)

### 4. Triggers
- `processingTime` : Intervalles réguliers (10s)
- Autres : `once`, `continuous`

### 5. Delta Lake (Bronze)
- Transactions ACID
- Versioning et time travel
- Données brutes avec métadonnées

### 6. Architecture Médaillon
- **Bronze** : Stockage brut avec validation minimale
- Source de vérité pour Silver/Gold